<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">Las Consideraciones de Rendimiento</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">Optimizar inferencia obliga a pensar en latencia, memoria, throughput y coste computacional como parte del diseño del sistema. Este notebook cierra la fase poniendo foco en esos criterios prácticos.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Un modelo útil en producción se evalúa también por cómo responde bajo restricciones reales de rendimiento.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A interpretar el rendimiento como una dimensión técnica que debe equilibrarse con la calidad del modelo.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">Mejor rendimiento no significa solo ejecutar más rápido; significa rendir mejor bajo las restricciones que importan en tu contexto.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">Analizar rendimiento implica observar tiempos, recursos y compromisos entre eficiencia y exactitud.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Carga del modelo y del tokenizer
"""Este bloque carga un modelo preentrenado y su tokenizer asociado.
Este bloque importa "AutoModelForSequenceClassification" y "AutoTokenizer" de "transformers" para cargar un modelo preentrenado 
y su tokenizador correspondiente.

Aplicamos la clase "AutoModelForSequenceClassification", acompañada del método ".from_pretrained()", a la variable "modelo" para cargar un modelo preentrenado.
El argumento "bert-base-uncased" especifica el modelo preentrenado que se desea utilizar.
El parámetro "num_labels=" especifica el número de etiquetas de clasificación que el modelo debe predecir.
La variable "modelo" almacena una instancia del modelo para clasificación de secuencias. Su cabeza de clasificación se inicializa de nuevo y no se entrena en este ejemplo.

Aplicamos la clase "AutoTokenizer", acompañada del método ".from_pretrained()", a la variable "tokenizador" para cargar un tokenizador preentrenado.
El argumento "bert-base-uncased" especifica el modelo preentrenado que se desea utilizar.
La variable "tokenizador" almacena una instancia del tokenizador preentrenado lista para ser utilizada en tareas de clasificación de secuencias.
Estos son los componentes fundamentales para evaluar consideraciones de rendimiento.
El modelo y el tokenizador están preentrenados para inglés, mientras que los textos del ejemplo están en español."""

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

modelo = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
tokenizador = AutoTokenizer.from_pretrained("bert-base-uncased")

modelo, tokenizador # Muestra el modelo y el tokenizador cargados.


In [ ]:
# Bloque 2 — Preparación de entradas para medir rendimiento
"""Este bloque define varias entradas de texto para evaluar el comportamiento del modelo con un batch.
La variable "textos" contiene múltiples secuencias que serán procesadas por el tokenizador en batch.

Asignamos la llamada a la instancia de la clase "tokenizador()" a la variable "batch_tokenizado" mediante la expresión "tokenizador(textos, padding=True, truncation=True, return_tensors="pt")".
El argumento "textos" indica el conjunto de textos que se desean tokenizar.
El parámetro "padding=" indica que se deben rellenar los textos para que tengan la longitud del más largo del batch.
El parámetro "truncation=" indica que se deben truncar los textos que excedan la longitud máxima permitida.
El parámetro "return_tensors=" indica con el valor "pt" que se debe devolver el resultado en formato tensor de PyTorch.
La variable "batch_tokenizado" almacena la versión tokenizada en formato tensor."""

textos = [
    "El servicio fue excelente.",
    "La experiencia fue terrible.",
    "El producto llegó rápido.",
    "La calidad es deficiente.",
    "El soporte técnico fue útil."
]

batch_tokenizado = tokenizador(textos, padding=True, truncation=True, return_tensors="pt")

batch_tokenizado # Muestra la versión tokenizada en formato tensor de los textos.


In [ ]:
# Bloque 3 — Medición de latencia de inferencia estándar
"""Este bloque mide la latencia de inferencia estándar sin desactivar el cálculo de gradientes.
Este bloque importa "time" para trabajar con mediciones de tiempo y calcular la latencia de inferencia en PyTorch.

Aplicamos la función "time.time()" a la variable "tiempo_inicio" antes de la inferencia para obtener el tiempo de inicio.
Asignamos el atributo ".logits" de la salida de "modelo()" a la variable "logits_estandar".
El argumento "**batch_tokenizado" indica los tensores tokenizados que se pasan al modelo.
Los asteriscos dobles "**" se utilizan para desempaquetar el diccionario "batch_tokenizado" y pasar sus elementos como argumentos al modelo.
La variable "logits_estandar" contiene las salidas del modelo antes de realizar la operación argmax.
Aplicamos la función "time.time()" a la variable "tiempo_fin" después de la inferencia para obtener el tiempo de finalización.

Restamos el tiempo de inicio al tiempo de finalización mediante la fórmula (tiempo_fin - tiempo_inicio) para obtener la latencia de inferencia estándar.
La variable "latencia_estandar" almacena la duración total de la inferencia estándar.
Es una medición única sin calentamiento ni sincronización explícita de GPU, por lo que no constituye una comparación fiable de rendimiento."""

import time

tiempo_inicio = time.time()
logits_estandar = modelo(**batch_tokenizado).logits
tiempo_fin = time.time()

latencia_estandar = (tiempo_fin - tiempo_inicio)

latencia_estandar # Muestra la latencia de inferencia estándar para el batch tokenizado actual.


In [ ]:
# Bloque 4 — Activación de modo no_grad para optimización
"""Este bloque activa el modo no_grad para reducir el uso de memoria y, según el caso, acelerar la inferencia.
Aplicamos la función "time.time()" a la variable "tiempo_inicio" antes de la inferencia para obtener el tiempo de inicio.

Aplicamos la clase "torch.no_grad()" mediante un bloque "with" para crear un entorno de gradientes desactivados durante la inferencia.
Dentro del entorno no_grad, asignamos el atributo ".logits" de la salida de "modelo()" a la variable "logits_no_grad".
El argumento "**batch_tokenizado" indica los tensores tokenizados que se pasan al modelo en el entorno no_grad.
Los asteriscos dobles "**" se utilizan para desempaquetar el diccionario "batch_tokenizado" y pasar sus elementos como argumentos al modelo.
La variable "logits_no_grad" contiene las salidas del modelo sin cálculo de gradientes.

Aplicamos la función "time.time()" a la variable "tiempo_fin" después de la inferencia para obtener el tiempo de finalización.
Restamos el tiempo de inicio al tiempo de finalización mediante la fórmula (tiempo_fin - tiempo_inicio) para obtener la latencia sin gradientes.
La variable "latencia_no_grad" almacena la duración de la inferencia sin cálculo de gradientes."""

tiempo_inicio = time.time()

with torch.no_grad():
    logits_no_grad = modelo(**batch_tokenizado).logits

tiempo_fin = time.time()

latencia_no_grad = (tiempo_fin - tiempo_inicio)
 
latencia_no_grad # Muestra la latencia de inferencia sin cálculo de gradientes para el batch tokenizado.


In [ ]:
# Bloque 5 — Cuantización dinámica del modelo
"""Este bloque aplica cuantización dinámica al modelo con el objetivo de reducir su tamaño y mejorar la inferencia.
Este bloque importa "copy" para crear una copia del modelo en memoria antes de aplicar la cuantización dinámica.
Este bloque importa "Int8DynamicActivationInt8WeightConfig" y "quantize_" de "torchao.quantization" para aplicar la cuantización dinámica al modelo.

Aplicamos la función "copy.deepcopy()" al modelo para crear una copia en memoria antes de aplicar la cuantización dinámica.
La variable "modelo_cuantizado" almacena la copia del modelo que será optimizada mediante cuantización dinámica.

Aplicamos la función "quantize_()" al modelo copiado y a la clase "Int8DynamicActivationInt8WeightConfig()" para realizar la cuantización dinámica.
Esta operación cuantiza las capas compatibles del modelo copiado sin modificar el modelo original; su efecto en el rendimiento depende del entorno.
La variable "modelo_cuantizado" almacena ahora la versión cuantizada del modelo copiado, lista para inferencia."""

import copy
from torchao.quantization import (Int8DynamicActivationInt8WeightConfig, quantize_,)

modelo_cuantizado = copy.deepcopy(modelo)
quantize_(modelo_cuantizado, Int8DynamicActivationInt8WeightConfig())

# Esta operación cuantiza las capas compatibles del modelo copiado sin modificar el modelo original.


In [ ]:
# Bloque 6 — Medición de latencia con modelo cuantizado
"""Este bloque mide la latencia de inferencia utilizando el modelo cuantizado.
Aplicamos la función "time.time()" a la variable "tiempo_inicio" antes de la inferencia para obtener el tiempo de inicio.

Asignamos el atributo ".logits" de la salida de "modelo_cuantizado()" a la variable "logits_cuantizado".
El argumento "**batch_tokenizado" indica los tensores tokenizados que se pasan al modelo.
Los asteriscos dobles "**" se utilizan para desempaquetar el diccionario "batch_tokenizado" y pasar sus elementos como argumentos al modelo.
La variable "logits_cuantizado" contiene las salidas del modelo después de la inferencia.
Aplicamos la función "time.time()" a la variable "tiempo_fin" después de la inferencia para obtener el tiempo de finalización.

Restamos el tiempo de inicio al tiempo de finalización mediante la fórmula (tiempo_fin - tiempo_inicio) para obtener la latencia de inferencia cuantizada.
La variable "latencia_cuantizado" almacena la duración total de la inferencia cuantizada."""

tiempo_inicio = time.time()

with torch.no_grad():
    logits_cuantizado = modelo_cuantizado(**batch_tokenizado).logits

tiempo_fin = time.time()

latencia_cuantizado = (tiempo_fin - tiempo_inicio)

latencia_cuantizado # Mostramos la latencia de inferencia cuantizada.


In [ ]:
# Bloque 7 — Comparación del tamaño serializado
"""Este bloque compara el tamaño en bytes de los pesos serializados con PyTorch.
Este bloque importa "tempfile" para crear archivos temporales donde se guardan los pesos del modelo.
Este bloque importa "torch" para trabajar con tensores y guardar los pesos del modelo.

Definimos la función "def tamaño_modelo(modelo)" que calcula el tamaño en bytes del estado serializado de un modelo dado.
El argumento "modelo" es el modelo de PyTorch cuyo tamaño de pesos queremos calcular.

Creamos un entorno temporal para guardar los pesos del modelo y calcular su tamaño.
La expresión "with tempfile.TemporaryFile() as archivo:" crea un archivo temporal llamado "archivo" donde se guardan los pesos del modelo.

Dentro del bloque "with", guardamos los pesos del modelo en el archivo temporal y luego obtenemos el tamaño en bytes del archivo.
La expresión "torch.save(modelo.state_dict(), archivo)" guarda los pesos del modelo en el archivo temporal.

Aplicamos el método ".flush()" al archivo temporal para asegurar que los datos almacenados en el búfer se escriban antes de medir su tamaño.
Retornamos el tamaño en bytes del archivo temporal usando "os.fstat(archivo.fileno()).st_size".
Los archivos temporales se eliminan al terminar; no se guardan modelos para su distribución.

Aplicamos la función "tamaño_modelo" a los modelos original y cuantizado para obtener sus tamaños en bytes correspondientes.
En cada caso, las variables "tamaño_original" y "tamaño_cuantizado" almacenan los tamaños serializados de los modelos original y cuantizado, respectivamente.
Esta comparación no mide directamente la memoria utilizada durante la inferencia."""

import os
import tempfile

def tamaño_modelo(modelo):
    with tempfile.TemporaryFile() as archivo:
        torch.save(modelo.state_dict(), archivo)
        archivo.flush()
        return os.fstat(archivo.fileno()).st_size


tamaño_original = tamaño_modelo(modelo)
tamaño_cuantizado = tamaño_modelo(modelo_cuantizado)



In [ ]:
# Bloque 8 — Resultados finales
"""Este bloque organiza los resultados del análisis de rendimiento.
La variable "resultado" almacena en forma de diccionario las latencias medidas y los tamaños serializados del modelo."""

resultado = {
    "latencia_estandar": latencia_estandar,
    "latencia_no_grad": latencia_no_grad,
    "latencia_cuantizado": latencia_cuantizado,
    "tamaño_original_bytes": tamaño_original,
    "tamaño_cuantizado_bytes": tamaño_cuantizado
}

resultado # Muestra el diccionario con los resultados finales de latencias y tamaños del modelo.


<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto que el rendimiento es una dimensión de diseño tan real como la arquitectura o la métrica final.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">Desplegar bien un modelo exige equilibrar precisión, coste y velocidad según el contexto de uso.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Optimizar un único indicador de rendimiento sin mirar el efecto conjunto sobre el sistema completo.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en sala a para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en sala a y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">05_LABORATORIO_/01_Sala_a.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">